In [72]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import scipy.stats as ss
import seaborn as sns
from pathlib import Path
import dcor
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder

In [73]:
DATA_DIR = Path('../data')

In [74]:
df = pd.read_csv(DATA_DIR / 'raw' / 'hotel_bookings.csv', encoding='utf-8')
df.head()

,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,...,deposit_type,agent,company,days_in_waiting_list,customer_type,adr,required_car_parking_spaces,total_of_special_requests,reservation_status,reservation_status_date
0,Resort Hotel,0,342,2015,July,27,1,0,0,2,...,No Deposit,NaN,NaN,0,Transient,0.0,0,0,Check-Out,2015-07-01
1,Resort Hotel,0,737,2015,July,27,1,0,0,2,...,No Deposit,NaN,NaN,0,Transient,0.0,0,0,Check-Out,2015-07-01
2,Resort Hotel,0,7,2015,July,27,1,0,1,1,...,No Deposit,NaN,NaN,0,Transient,75.0,0,0,Check-Out,2015-07-02
3,Resort Hotel,0,13,2015,July,27,1,0,1,1,...,No Deposit,304.0,NaN,0,Transient,75.0,0,0,Check-Out,2015-07-02
4,Resort Hotel,0,14,2015,July,27,1,0,2,2,...,No Deposit,240.0,NaN,0,Transient,98.0,0,1,Check-Out,2015-07-03


In [75]:
df.shape

(119390, 32)

In [76]:
df.isna().sum()

hotel                                  0
is_canceled                            0
lead_time                              0
arrival_date_year                      0
arrival_date_month                     0
arrival_date_week_number               0
arrival_date_day_of_month              0
stays_in_weekend_nights                0
stays_in_week_nights                   0
adults                                 0
children                               4
babies                                 0
meal                                   0
country                              488
market_segment                         0
distribution_channel                   0
is_repeated_guest                      0
previous_cancellations                 0
previous_bookings_not_canceled         0
reserved_room_type                     0
assigned_room_type                     0
booking_changes                        0
deposit_type                           0
agent                              16340
company         

Видим небольшое количество пропусков, значительное количество в тур-агентах и компаниях-сопроводителях. Скорее всего это значит, что бронь оформляло физ.лицо, для этого можно выделить отдельную категорию

# Описание признаков

## Результат бронирования

- **`is_canceled`** — было ли бронирование отменено: `1` — да, `0` — нет.
- **`reservation_status`** — последний статус бронирования:
  - `Canceled` — отменено;
  - `Check-Out` — гость выехал;
  - `No-Show` — гость не приехал и не сообщил причину.
- **`reservation_status_date`** — дата установки последнего статуса бронирования.

## Отель и дата заезда

- **`hotel`** — тип отеля: `Resort Hotel` — курортный, `City Hotel` — городской.
- **`arrival_date_year`** — год заезда.
- **`arrival_date_month`** — месяц заезда.
- **`arrival_date_week_number`** — номер недели заезда в году.
- **`arrival_date_day_of_month`** — день месяца, на который приходится заезд.
- **`lead_time`** — количество дней между внесением бронирования в PMS и датой заезда.
- **`stays_in_weekend_nights`** — количество выходных ночей (суббота или воскресенье), забронированных или проведённых в отеле.
- **`stays_in_week_nights`** — количество будних ночей (с понедельника по пятницу), забронированных или проведённых в отеле.

## Гость и история бронирований

- **`adults`** — количество взрослых.
- **`children`** — количество детей.
- **`babies`** — количество младенцев.
- **`country`** — страна происхождения гостя. Используются коды стандарта ISO 3155-3:2013.
- **`is_repeated_guest`** — является ли гость повторным: `1` — да, `0` — нет.
- **`previous_cancellations`** — количество предыдущих бронирований клиента, отменённых до текущего.
- **`previous_bookings_not_canceled`** — количество предыдущих бронирований клиента, которые не были отменены.
- **`customer_type`** — тип бронирования:
  - `Contract` — связано с договором или выделенной квотой номеров;
  - `Group` — связано с группой;
  - `Transient` — индивидуальное бронирование, не связанное с группой или договором;
  - `Transient-party` — индивидуальное бронирование, связанное с другими индивидуальными бронированиями.

## Номер и условия проживания

- **`reserved_room_type`** — обезличенный код забронированного типа номера.
- **`assigned_room_type`** — обезличенный код назначенного типа номера. Может отличаться от забронированного из-за особенностей работы отеля или пожеланий гостя.
- **`stays_in_weekend_nights`** — количество выходных ночей проживания.
- **`stays_in_week_nights`** — количество будних ночей проживания.
- **`adr`** — средняя стоимость проживания за ночь. Рассчитывается как сумма платежей за проживание, разделённая на общее количество ночей.
- **`required_car_parking_spaces`** — количество запрошенных парковочных мест.
- **`total_of_special_requests`** — количество особых пожеланий гостя, например отдельные кровати или номер на высоком этаже.

## Канал бронирования и оплата

- **`meal`** — тип питания:
  - `Undefined` / `SC` — питание не включено;
  - `BB` — завтрак;
  - `HB` — полупансион;
  - `FB` — полный пансион.
- **`market_segment`** — сегмент рынка. `TA` — туристические агентства, `TO` — туроператоры.
- **`distribution_channel`** — канал оформления бронирования. `TA` — туристические агентства, `TO` — туроператоры.
- **`deposit_type`** — условия внесения депозита:
  - `No Deposit` — без депозита;
  - `Non Refund` — невозвратный депозит;
  - `Refundable` — возвратный депозит.
- **`agent`** — идентификатор туристического агентства, оформившего бронирование.
- **`company`** — идентификатор компании или организации, оформившей или оплатившей бронирование.

## Изменения бронирования и лист ожидания

- **`booking_changes`** — количество изменений бронирования с момента его внесения в PMS до заезда или отмены.
- **`days_in_waiting_list`** — количество дней, в течение которых бронирование находилось в листе ожидания до подтверждения.

> **PMS** (*Property Management System*) — система управления отелем.

In [77]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 119390 entries, 0 to 119389
Data columns (total 32 columns):
 #   Column                          Non-Null Count   Dtype  
---  ------                          --------------   -----  
 0   hotel                           119390 non-null  str    
 1   is_canceled                     119390 non-null  int64  
 2   lead_time                       119390 non-null  int64  
 3   arrival_date_year               119390 non-null  int64  
 4   arrival_date_month              119390 non-null  str    
 5   arrival_date_week_number        119390 non-null  int64  
 6   arrival_date_day_of_month       119390 non-null  int64  
 7   stays_in_weekend_nights         119390 non-null  int64  
 8   stays_in_week_nights            119390 non-null  int64  
 9   adults                          119390 non-null  int64  
 10  children                        119386 non-null  float64
 11  babies                          119390 non-null  int64  
 12  meal                       

In [78]:
df.describe()

,is_canceled,lead_time,arrival_date_year,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,children,babies,is_repeated_guest,previous_cancellations,previous_bookings_not_canceled,booking_changes,agent,company,days_in_waiting_list,adr,required_car_parking_spaces,total_of_special_requests
count,119390.000000,119390.000000,119390.000000,119390.000000,119390.000000,119390.000000,119390.000000,119390.000000,119386.000000,119390.000000,119390.000000,119390.000000,119390.000000,119390.000000,103050.000000,6797.000000,119390.000000,119390.000000,119390.000000,119390.000000
mean,0.370416,104.011416,2016.156554,27.165173,15.798241,0.927599,2.500302,1.856403,0.103890,0.007949,0.031912,0.087118,0.137097,0.221124,86.693382,189.266735,2.321149,101.831122,0.062518,0.571363
std,0.482918,106.863097,0.707476,13.605138,8.780829,0.998613,1.908286,0.579261,0.398561,0.097436,0.175767,0.844336,1.497437,0.652306,110.774548,131.655015,17.594721,50.535790,0.245291,0.792798
min,0.000000,0.000000,2015.000000,1.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,6.000000,0.000000,-6.380000,0.000000,0.000000
25%,0.000000,18.000000,2016.000000,16.000000,8.000000,0.000000,1.000000,2.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,9.000000,62.000000,0.000000,69.290000,0.000000,0.000000
50%,0.000000,69.000000,2016.000000,28.000000,16.000000,1.000000,2.000000,2.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,14.000000,179.000000,0.000000,94.575000,0.000000,0.000000
75%,1.000000,160.000000,2017.000000,38.000000,23.000000,2.000000,3.000000,2.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,229.000000,270.000000,0.000000,126.000000,0.000000,1.000000
max,1.000000,737.000000,2017.000000,53.000000,31.000000,19.000000,50.000000,55.000000,10.000000,10.000000,1.000000,26.000000,72.000000,21.000000,535.000000,543.000000,391.000000,5400.000000,8.000000,5.000000


In [79]:
df.select_dtypes(include=('Int64', 'Float64')).corr()['is_canceled']

is_canceled                       1.000000
lead_time                         0.293123
arrival_date_year                 0.016660
arrival_date_week_number          0.008148
arrival_date_day_of_month        -0.006130
stays_in_weekend_nights          -0.001791
stays_in_week_nights              0.024765
adults                            0.060017
children                          0.005048
babies                           -0.032491
is_repeated_guest                -0.084793
previous_cancellations            0.110133
previous_bookings_not_canceled   -0.057358
booking_changes                  -0.144381
agent                            -0.083114
company                          -0.020642
days_in_waiting_list              0.054186
adr                               0.047557
required_car_parking_spaces      -0.195498
total_of_special_requests        -0.234658
Name: is_canceled, dtype: float64

In [80]:
df_dcor_matrix = pd.Series()

for col in df.select_dtypes(include=['Int64', 'Float32']):
    df_dcor_matrix[col] = dcor.distance_correlation(df[col], df['is_canceled'])

df_dcor_matrix

c:\Users\Max-W10\AppData\Local\Programs\Python\Python314\Lib\site-packages\dcor\_fast_dcov_avl.py:554: UserWarning: Falling back to uncompiled AVL fast distance covariance terms because of TypeError exception raised: No matching definition for argument type(s) readonly array(int64, 1d, C), readonly array(int64, 1d, C), bool. Rembember: only floating point values can be used in the compiled implementations.
  warnings.warn(


is_canceled                       1.000000
lead_time                         0.288619
arrival_date_year                 0.021913
arrival_date_week_number          0.030321
arrival_date_day_of_month         0.007903
stays_in_weekend_nights           0.007791
stays_in_week_nights              0.060078
adults                            0.080930
babies                            0.034305
is_repeated_guest                 0.084793
previous_cancellations            0.255449
previous_bookings_not_canceled    0.092664
booking_changes                   0.179467
days_in_waiting_list              0.077404
required_car_parking_spaces       0.197397
total_of_special_requests         0.254418
dtype: float64

In [81]:
non_numeric_cols = df.select_dtypes(exclude=['Int64', 'Float32']).columns
for col in non_numeric_cols:
    print(f'Название колонки: {col}\n{df[col].value_counts().sort_index()}')


Название колонки: hotel
hotel
City Hotel      79330
Resort Hotel    40060
Name: count, dtype: int64
Название колонки: arrival_date_month
arrival_date_month
April        11089
August       13877
December      6780
February      8068
January       5929
July         12661
June         10939
March         9794
May          11791
November      6794
October      11160
September    10508
Name: count, dtype: int64
Название колонки: children
children
0.0     110796
1.0       4861
2.0       3652
3.0         76
10.0         1
Name: count, dtype: int64
Название колонки: meal
meal
BB           92310
FB             798
HB           14463
SC           10650
Undefined     1169
Name: count, dtype: int64
Название колонки: country
country
ABW      2
AGO    362
AIA      1
ALB     12
AND      7
      ... 
VGB      1
VNM      8
ZAF     80
ZMB      2
ZWE      4
Name: count, Length: 177, dtype: int64
Название колонки: market_segment
market_segment
Aviation           237
Complementary      743
Corporate       

In [82]:
df_prep = df.copy()

In [83]:
str_to_int = ['children', 'agent', 'company']
df_prep[str_to_int] = (
    df_prep[str_to_int]
    .apply(pd.to_numeric, errors='coerce')
    .fillna(0).astype('Int64')
)

# !

Видим, что значения в этих колонках имели строковые значения, хотя в своей идее должны быть целочисленными (кол-во детей, ID агентов или компаний). Заполнил 0, так как при отсутствии указывания кол-ва детей скорее всего значит, что взрослые туристы не заполнили эту графу за ненадобностью, либо значение упущено, в любом случае 0 также считается модой. ID агентов и компаний = 0, в том случае, когда бронирование произошло без них

In [84]:
df_prep[df_prep.values == 'Undefined'] = np.nan
df_prep.isna().sum() > 0

hotel                             False
is_canceled                       False
lead_time                         False
arrival_date_year                 False
arrival_date_month                False
arrival_date_week_number          False
arrival_date_day_of_month         False
stays_in_weekend_nights           False
stays_in_week_nights              False
adults                            False
children                          False
babies                            False
meal                               True
country                            True
market_segment                     True
distribution_channel               True
is_repeated_guest                 False
previous_cancellations            False
previous_bookings_not_canceled    False
reserved_room_type                False
assigned_room_type                False
booking_changes                   False
deposit_type                      False
agent                             False
company                           False


# !

Замена метки Undefined на NaN, можно было это реализовать при чтении csv файла, раз уж заметил такой формат записи пропуска, несложно исправить это сейчас

In [85]:
df_prep.loc[:, (df_prep.isna().sum() > 0).values].columns

Index(['meal', 'country', 'market_segment', 'distribution_channel'], dtype='str')

Проверил, что пропуски остались только в странах

In [86]:
df_prep.dtypes

hotel                                 str
is_canceled                         int64
lead_time                           int64
arrival_date_year                   int64
arrival_date_month                    str
arrival_date_week_number            int64
arrival_date_day_of_month           int64
stays_in_weekend_nights             int64
stays_in_week_nights                int64
adults                              int64
children                            Int64
babies                              int64
meal                                  str
country                               str
market_segment                        str
distribution_channel                  str
is_repeated_guest                   int64
previous_cancellations              int64
previous_bookings_not_canceled      int64
reserved_room_type                    str
assigned_room_type                    str
booking_changes                     int64
deposit_type                          str
agent                             

In [87]:
df_prep['arrival_date_month'].unique()

<StringArray>
[     'July',    'August', 'September',   'October',  'November',  'December',
   'January',  'February',     'March',     'April',       'May',      'June']
Length: 12, dtype: str

In [88]:
months = [    
    'January', 'February', 'March', 'April', 'May', 'June', 'July',
    'August', 'September', 'October', 'November', 'December'
]

encoder = OrdinalEncoder(categories=[months])

df_prep['arrival_date_month'] = encoder.fit_transform(df[['arrival_date_month']])
df_prep['arrival_date_month'] = df_prep['arrival_date_month'].astype('Int64') + 1

df_prep['arrival_date_month'].unique()

<IntegerArray>
[7, 8, 9, 10, 11, 12, 1, 2, 3, 4, 5, 6]
Length: 12, dtype: Int64

# !
label-encoding месяцев заезда

In [89]:
df_prep.isna().sum()

hotel                                0
is_canceled                          0
lead_time                            0
arrival_date_year                    0
arrival_date_month                   0
arrival_date_week_number             0
arrival_date_day_of_month            0
stays_in_weekend_nights              0
stays_in_week_nights                 0
adults                               0
children                             0
babies                               0
meal                              1169
country                            488
market_segment                       2
distribution_channel                 5
is_repeated_guest                    0
previous_cancellations               0
previous_bookings_not_canceled       0
reserved_room_type                   0
assigned_room_type                   0
booking_changes                      0
deposit_type                         0
agent                                0
company                              0
days_in_waiting_list     

In [90]:
df_prep['country'].value_counts()

country
PRT    48590
GBR    12129
FRA    10415
ESP     8568
DEU     7287
       ...  
NCL        1
KIR        1
SDN        1
ATF        1
SLE        1
Name: count, Length: 177, dtype: int64

In [91]:
imputer = SimpleImputer(strategy='most_frequent')
df_prep['country'] = imputer.fit_transform(df_prep[['country']]).ravel()
df_prep['country'].isna().sum()

np.int64(0)

In [92]:
for col in df_prep.select_dtypes(include=['str']).columns:
    print(f'Столбец: {col}\n {df_prep[col].value_counts()}')

Столбец: hotel
 hotel
City Hotel      79330
Resort Hotel    40060
Name: count, dtype: int64
Столбец: meal
 meal
BB    92310
HB    14463
SC    10650
FB      798
Name: count, dtype: int64
Столбец: country
 country
PRT    49078
GBR    12129
FRA    10415
ESP     8568
DEU     7287
       ...  
NCL        1
KIR        1
SDN        1
ATF        1
SLE        1
Name: count, Length: 177, dtype: int64
Столбец: market_segment
 market_segment
Online TA        56477
Offline TA/TO    24219
Groups           19811
Direct           12606
Corporate         5295
Complementary      743
Aviation           237
Name: count, dtype: int64
Столбец: distribution_channel
 distribution_channel
TA/TO        97870
Direct       14645
Corporate     6677
GDS            193
Name: count, dtype: int64
Столбец: reserved_room_type
 reserved_room_type
A    85994
D    19201
E     6535
F     2897
G     2094
B     1118
C      932
H      601
P       12
L        6
Name: count, dtype: int64
Столбец: assigned_room_type
 assigned_roo

In [93]:
df_prep.loc[df_prep['meal'].isna(), 'meal'] = 'SC'
df_prep['meal'].value_counts()

meal
BB    92310
HB    14463
SC    11819
FB      798
Name: count, dtype: int64

# !

SC и Undefined значили то же самое по своей идейной составляющей, легче заменить на одну категорию для дальнейшего Label-encoding

In [94]:
nan_to_most_fr = ['market_segment', 'distribution_channel']

imputer2 = SimpleImputer(strategy='most_frequent')
df_prep[nan_to_most_fr] = imputer2.fit_transform(df_prep[nan_to_most_fr])
df_prep.isna().sum()

hotel                             0
is_canceled                       0
lead_time                         0
arrival_date_year                 0
arrival_date_month                0
arrival_date_week_number          0
arrival_date_day_of_month         0
stays_in_weekend_nights           0
stays_in_week_nights              0
adults                            0
children                          0
babies                            0
meal                              0
country                           0
market_segment                    0
distribution_channel              0
is_repeated_guest                 0
previous_cancellations            0
previous_bookings_not_canceled    0
reserved_room_type                0
assigned_room_type                0
booking_changes                   0
deposit_type                      0
agent                             0
company                           0
days_in_waiting_list              0
customer_type                     0
adr                         

# !

Замена пропусков на моду, можем себе это позволить, так как доля пропусков ничтожно мала по отношению к общему числу строк датафрейма, статистически мы ничего не изменили

In [95]:
df_prep['reservation_status_date'] = pd.to_datetime(
    df_prep['reservation_status_date'],
    errors='coerce',
    format='%Y-%m-%d'
)
df_prep['reservation_status_date'].dtypes

dtype('<M8[us]')

# !

Привели колонку дат в нужный формат

In [96]:
non_numeric_cols = df_prep.select_dtypes(include=['object', 'str']).columns

encoder2 = OrdinalEncoder(
    handle_unknown='use_encoded_value',
    unknown_value=-1,
    encoded_missing_value=-1
)
df_prep[non_numeric_cols] = encoder2.fit_transform(df_prep[non_numeric_cols]).astype(np.int64)
df_prep.dtypes

hotel                                      int64
is_canceled                                int64
lead_time                                  int64
arrival_date_year                          int64
arrival_date_month                         Int64
arrival_date_week_number                   int64
arrival_date_day_of_month                  int64
stays_in_weekend_nights                    int64
stays_in_week_nights                       int64
adults                                     int64
children                                   Int64
babies                                     int64
meal                                       int64
country                                    int64
market_segment                             int64
distribution_channel                       int64
is_repeated_guest                          int64
previous_cancellations                     int64
previous_bookings_not_canceled             int64
reserved_room_type                         int64
assigned_room_type  

# !

Финальный label-encoding

In [97]:
df_prep_dcor_matrix = pd.Series()

for col in df_prep.columns:
    df_prep_dcor_matrix[col] = dcor.distance_correlation(df_prep[col], df_prep['is_canceled'])

df_prep_dcor_matrix

c:\Users\Max-W10\AppData\Local\Programs\Python\Python314\Lib\site-packages\dcor\_fast_dcov_avl.py:554: UserWarning: Falling back to uncompiled AVL fast distance covariance terms because of TypeError exception raised: No matching definition for argument type(s) readonly array(int64, 1d, C), readonly array(int64, 1d, C), bool. Rembember: only floating point values can be used in the compiled implementations.
  warnings.warn(
c:\Users\Max-W10\AppData\Local\Programs\Python\Python314\Lib\site-packages\dcor\_fast_dcov_avl.py:554: UserWarning: Falling back to uncompiled AVL fast distance covariance terms because of TypeError exception raised: No matching definition for argument type(s) readonly array(int64, 1d, C), readonly array(int64, 1d, C), bool. Rembember: only floating point values can be used in the compiled implementations.
  warnings.warn(
c:\Users\Max-W10\AppData\Local\Programs\Python\Python314\Lib\site-packages\dcor\_fast_dcov_avl.py:554: UserWarning: Falling back to uncompiled AVL

hotel                             0.136531
is_canceled                       1.000000
lead_time                         0.288619
arrival_date_year                 0.021913
arrival_date_month                0.029823
arrival_date_week_number          0.030321
arrival_date_day_of_month         0.007903
stays_in_weekend_nights           0.007791
stays_in_week_nights              0.060078
adults                            0.080930
children                          0.011167
babies                            0.034305
meal                              0.016857
country                           0.292701
market_segment                    0.106136
distribution_channel              0.173335
is_repeated_guest                 0.084793
previous_cancellations            0.255449
previous_bookings_not_canceled    0.092664
reserved_room_type                0.067446
assigned_room_type                0.187524
booking_changes                   0.179467
deposit_type                      0.477865
agent      

In [104]:
for col in df_prep_dcor_matrix.index:
    if col not in df_dcor_matrix:
        df_dcor_matrix[col] = 0

for col in df_prep_dcor_matrix.index:
    print(f'Не обработанный {col}: {np.round(df_dcor_matrix[col], 3)}\nОбработанный {col}: {np.round(df_prep_dcor_matrix[col], 3)}\n')

Не обработанный hotel: 0.0
Обработанный hotel: 0.137

Не обработанный is_canceled: 1.0
Обработанный is_canceled: 1.0

Не обработанный lead_time: 0.289
Обработанный lead_time: 0.289

Не обработанный arrival_date_year: 0.022
Обработанный arrival_date_year: 0.022

Не обработанный arrival_date_month: 0.0
Обработанный arrival_date_month: 0.03

Не обработанный arrival_date_week_number: 0.03
Обработанный arrival_date_week_number: 0.03

Не обработанный arrival_date_day_of_month: 0.008
Обработанный arrival_date_day_of_month: 0.008

Не обработанный stays_in_weekend_nights: 0.008
Обработанный stays_in_weekend_nights: 0.008

Не обработанный stays_in_week_nights: 0.06
Обработанный stays_in_week_nights: 0.06

Не обработанный adults: 0.081
Обработанный adults: 0.081

Не обработанный children: 0.0
Обработанный children: 0.011

Не обработанный babies: 0.034
Обработанный babies: 0.034

Не обработанный meal: 0.0
Обработанный meal: 0.017

Не обработанный country: 0.0
Обработанный country: 0.293

Не обрабо